# GoEmotions — Experimental Multi-Label Evaluation

This notebook evaluates the **improved trained artifact** at `backend/models/goemotions_roberta_improved`. GoEmotions is an experimental/comparison model only; it is **not used by the MindCare frontend or production chat route**.

The notebook follows `app/ai/emotion_detection/goemotions_train_improved.py`: `roberta-base`, 28 named labels, multi-label sigmoid output, and a decision threshold of `0.30`. It does not evaluate the older single-label `goemotions_roberta` artifact.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BACKEND_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd() / 'backend'
if not (BACKEND_DIR / 'models').exists():
    BACKEND_DIR = Path.cwd().resolve()
MODEL_DIR = BACKEND_DIR / 'models' / 'goemotions_roberta_improved'
DATA_DIR = BACKEND_DIR / 'datasets' / 'processed_goemotions_multilabel'
OUTPUT_DIR = BACKEND_DIR / 'evaluation' / 'goemotions'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
THRESHOLD = 0.30
print('Model:', MODEL_DIR)
print('Test data:', DATA_DIR / 'clean_test.csv')
print('Threshold:', THRESHOLD)

## 28-label mapping

The mapping is copied from the improved training/preprocessing pipeline, not inferred from prediction order.

In [ ]:
LABELS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]
assert len(LABELS) == 28
label_to_id = {label: i for i, label in enumerate(LABELS)}
id_to_label = {i: label for i, label in enumerate(LABELS)}
LABELS

In [ ]:
def parse_multilabel(value):
    # Supports the CSV formats produced by the improved preprocessor:
    # comma-separated names, JSON/list strings, or integer IDs.
    if isinstance(value, list):
        raw = value
    else:
        text = str(value).strip()
        try:
            raw = json.loads(text) if text.startswith('[') else [x.strip() for x in text.split(',')]
        except json.JSONDecodeError:
            raw = [x.strip() for x in text.split(',')]
    vector = np.zeros(len(LABELS), dtype=np.float32)
    for item in raw:
        if str(item).isdigit() and int(item) < len(LABELS):
            vector[int(item)] = 1
        elif str(item) in label_to_id:
            vector[label_to_id[str(item)]] = 1
    return vector

test_df = pd.read_csv(DATA_DIR / 'clean_test.csv')
text_column = 'text' if 'text' in test_df.columns else 'sentence'
label_column = next(c for c in ['labels', 'label', 'emotion_labels'] if c in test_df.columns)
texts = test_df[text_column].astype(str).tolist()
targets = np.vstack(test_df[label_column].map(parse_multilabel).to_list())
print(test_df.shape, text_column, label_column)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def predict_multilabel(batch_size=32):
    probabilities = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(texts[start:start + batch_size], truncation=True, padding=True, max_length=128, return_tensors='pt')
        batch = {key: value.to(device) for key, value in batch.items()}
        with torch.no_grad():
            logits = model(**batch).logits
        probabilities.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(probabilities)

probabilities = predict_multilabel()
predictions = (probabilities >= THRESHOLD).astype(int)
print('Probability shape:', probabilities.shape)
print('Prediction shape:', predictions.shape)

In [ ]:
metrics = {
    'threshold': THRESHOLD,
    'samples': int(len(texts)),
    'labels': len(LABELS),
    'micro_f1': float(f1_score(targets, predictions, average='micro', zero_division=0)),
    'macro_f1': float(f1_score(targets, predictions, average='macro', zero_division=0)),
    'weighted_f1': float(f1_score(targets, predictions, average='weighted', zero_division=0)),
    'micro_precision': float(precision_score(targets, predictions, average='micro', zero_division=0)),
    'micro_recall': float(recall_score(targets, predictions, average='micro', zero_division=0)),
}
report = classification_report(targets, predictions, target_names=LABELS, zero_division=0, output_dict=True)
metrics['per_label'] = report
with open(OUTPUT_DIR / 'goemotions_improved_metrics_notebook.json', 'w', encoding='utf-8') as handle:
    json.dump(metrics, handle, indent=2)
pd.DataFrame(report).T.to_csv(OUTPUT_DIR / 'goemotions_improved_classification_report.csv')
pd.DataFrame({'text': texts, 'true_labels': [', '.join(LABELS[i] for i, v in enumerate(row) if v) for row in targets], 'predicted_labels': [', '.join(LABELS[i] for i, v in enumerate(row) if v) for row in predictions]}).to_csv(OUTPUT_DIR / 'goemotions_improved_predictions.csv', index=False)
print(json.dumps({k: v for k, v in metrics.items() if k != 'per_label'}, indent=2))

## Production boundary

These results are for research comparison only. The production chat route uses the DAIR-AI emotion classifier, stress classifier, and depression classifier. GoEmotions is not imported by the frontend and must not be described as a live clinical prediction model.